# Wavebit Tutorial 7 — arriving on more than one order

Every other incidence keyword describes **one** plane wave: `theta` and `phi` its
direction, `psi` / `polarization` / `tetm` its polarization. `excitation` describes
*several at once* — light arriving on more than one diffraction order, coherently, in a
single solve.

This notebook covers, in order:

1. what the default already is, written out
2. the keyword
3. **the trap** — one order on its own is not a new problem
4. the case it is actually for, and what it saves
5. the one thing it refuses
6. a different polarization per order, which goes through a different door

The measurements and the reasoning behind the design are in
[`../docs/multimode-incidence.md`](../docs/multimode-incidence.md).

In [ ]:
import time

import numpy as np

import wavebit

WL, PERIOD = 600.0, 2000.0          # lambda / period = 0.3, so several orders propagate
MAX_ORDER = 6                        # orders -6 .. +6 are retained: 13 of them


def grating():
    """Two layers on glass. A raster holds *relative permittivity*: 6.25 is n = 2.5."""
    eps = np.ones((2, 1, 64), dtype=complex)
    eps[1, 0, :26] = 6.25
    eps[0, 0, 20:] = 4.0
    return eps


def solve(excitation=None, theta=0.25, algorithm='ETM', **over):
    kw = dict(backend=0, truncation_order=(MAX_ORDER, 0), period=(PERIOD, PERIOD),
              wavelength=WL, theta=theta, phi=None, tetm='te',
              thickness=[130.0, 220.0], top=1.0, bot=2.25,
              layer_algorithm=algorithm, answers='amplitude', excitation=excitation)
    kw.update(over)
    solver = wavebit.rcwa(**kw)
    solver.structure = grating()
    return solver, solver.solve()

---

## 1. What the default already is

Left unset, `excitation` means *one wave, on the specular order*. That is the incident
column the solver has always built, and it can be written out:

In [ ]:
_, plain = solve()
_, spelled = solve({0: 1.0})            # the same thing, said explicitly

print('R, specular order only')
print('  unset          ', float(np.asarray(plain.incident.R).sum()))
print('  excitation={0:1}', float(np.asarray(spelled.incident.R).sum()))
print('  difference     ', abs(float(np.asarray(plain.incident.R).sum())
                              - float(np.asarray(spelled.incident.R).sum())))

The two agree, as they must. Leaving it unset is not a fast path but the definition:
the ordinary solve *is* the one-wave excitation.

(They agree to ~1e-16 rather than bit-for-bit. The undeclared path keeps the exact
expressions it always had, so that no stored result moves for a feature a scene does not
use; the declared one takes a general form that differs in the last bit.)

---

## 2. The keyword

A dict, keyed by diffraction order, valued by complex amplitude:

In [ ]:
BEAMS = {-1: 0.4 + 0.2j, 0: 1.0, 1: 0.6 - 0.8j}

three_solver, three = solve(BEAMS)

R = np.asarray(three.incident.R).ravel()
T = np.asarray(three.incident.T).ravel()
(centre, half), _ = three_solver.order_window
orders = np.arange(-half, half + 1) + centre

print(' order      R          T')
for m, r, t in zip(orders, R, T):
    if r > 1e-9 or t > 1e-9:
        print(f'  {m:+d}     {r:.6f}   {t:.6f}')
print(f'\n  R + T = {R.sum() + T.sum():.15f}')

Energy closes, which is the first thing to check: every efficiency is divided by the
**incident** flux, and for a superposition that is the sum over the lit orders rather than
one wave's. A denominator that counted only one of them would still look like an
efficiency.

Each weight multiplies **its own order's** polarization vector -- here `s_hat`, since
`tetm='te'` -- and on a 1D grating at `phi = 0` that is `sign(kx) * y_hat`. So the `-1` beam,
whose `kx` is negative at this angle, arrives polarized along `-y` while the other two are
along `+y`: exactly what `polarization=(1, 0)` lights, on every path. (The pure 1D path has
meant this since 2026-09-27; before, it put `+y` on every lit order, which is different light
and gave `sum(R) = 0.4933` here instead of `0.3701`. See
[`../docs/multimode-incidence.md`](../docs/multimode-incidence.md).)

On a 2D structure the keys are `(m, n)` pairs instead:

```python
wavebit.rcwa(..., excitation={(0, 0): 1.0, (1, 0): 0.5j})
```

---

## 3. The trap: one order on its own is just another angle

It is tempting to read `excitation={1: 1.0}` as "shine into the +1 order", something the
solver could not do before. It is not. The order index is a *label*, and which wave gets
which label depends on the angle you chose:

```
kx_m = n_inc * sin(theta) + m * (wavelength / period)
```

Lighting order `+1` at `theta` gives the same `kx` — the same plane wave — as lighting the
specular order at the angle one step along. Here they are, side by side:

In [ ]:
step = WL / PERIOD
theta_shifted = float(np.arcsin(step))       # kx of order +1 at theta = 0

print(f'lighting order +1 at theta = 0        -> kx = {step:.4f}')
print(f'lighting order  0 at theta = {theta_shifted:.4f}  -> kx = {np.sin(theta_shifted):.4f}')
print()


def by_order(solver, result):
    """Amplitudes keyed by diffraction order rather than by array index.

    An excitation widens the retained window, so `index = order + truncation_order` stops holding.
    `solver.order_window` reports what was actually kept.
    """
    (centre, half), _ = solver.order_window
    return dict(zip(np.arange(-half, half + 1) + centre,
                    np.asarray(result.incident.r_s).ravel()))


for truncation_order in (6, 10, 16, 24):
    a = by_order(*solve({1: 1.0}, theta=0.0, truncation_order=(truncation_order, 0)))
    b = by_order(*solve(None, theta=theta_shifted, truncation_order=(truncation_order, 0)))
    b = {order + 1: value for order, value in b.items()}   # b counts from its own specular
    trim = max(1, truncation_order // 3)
    shared = sorted(set(a) & set(b))[trim:-trim or None]
    lhs = np.array([a[k] for k in shared])
    rhs = np.array([b[k] for k in shared])
    err = np.abs(lhs - rhs).max() / np.abs(lhs).max()
    print(f'  truncation_order = {truncation_order:2d}   relative difference {err:.2e}')

About 1e-9 at every basis size — the two are one plane wave, and what is left is only that
the two solves keep slightly different finite windows of the same infinite ladder of orders.
(Compared by amplitude, the relabelling also needs each order's `kx` to be *the same number*
in both spellings: on this `ky = 0` row an amplitude is quoted along `sign(kx) * y_hat`, so an
order sitting at exactly `kx = 0` in one spelling and a rounding away from it in the other
would come back with the opposite sign. Here `sin(arcsin(0.3))` is exactly `0.3`, so none do.)

**So a single lit order buys a relabelling, not a new capability.** The value is in the
superposition, which no ordinary solve can express.

---

## 4. What it is for, and what it saves

The clear case is **coherent beam combining**: a grating is a beam splitter, and run
backwards it is a combiner — N beams arriving at the angles the grating equation fixes,
which is exactly "order 0 and order +1 at once".

You could do it without this keyword. The solve is linear in its incident field, so one
solve per beam and add the results:

```python
r_total  = a * solve(theta=theta_0).incident.r_s
r_total[1:] += b * solve(theta=theta_1).incident.r_s[:-1]      # shifted by one order
```

That works. What it costs is a solve per beam:

In [ ]:
def best_of(excitation, repeats=9):
    fastest = float('inf')
    for _ in range(repeats):
        solver = wavebit.rcwa(backend=0, truncation_order=(MAX_ORDER, 0), period=(PERIOD, PERIOD),
                              wavelength=WL, theta=0.25, phi=None, tetm='te',
                              thickness=[130.0, 220.0], top=1.0, bot=2.25,
                              answers='amplitude', excitation=excitation)
        solver.structure = grating()
        start = time.perf_counter()
        solver.solve()
        fastest = min(fastest, time.perf_counter() - start)
    return fastest


together = best_of(BEAMS)
apart = sum(best_of({order: amplitude}) for order, amplitude in BEAMS.items())

print(f'  three beams, one solve   {together * 1000:6.2f} ms')
print(f'  one solve per beam       {apart * 1000:6.2f} ms   ({apart / together:.2f}x)')

Roughly one solve per beam, as expected — plus two pieces of bookkeeping the caller no
longer performs, each of which is a quiet place to be wrong: computing every beam's angle,
and shifting each result by the right number of order indices before adding.

---

## 5. The window widens around the orders you light

A beam sitting at order +1 would, on the usual window, have one fewer neighbour above it
than below — while solving that beam alone would have centred it. Orders couple to their
neighbours by scattering, so that matters.

So the retained set widens: every lit order keeps `truncation_order` neighbours on each side.
`solver.order_window` reports what was actually kept, as `(centre, half_width)` per axis:

In [ ]:
# `order_window` is a property of the declaration, so nothing has to be solved to read it.
def window_of(excitation):
    solver = wavebit.rcwa(backend=0, truncation_order=(MAX_ORDER, 0), period=(PERIOD, PERIOD),
                          wavelength=WL, theta=0.25, phi=None, tetm='te',
                          thickness=[130.0, 220.0], top=1.0, bot=2.25,
                          excitation=excitation)
    (centre, half), _ = solver.order_window
    return centre, half


for label, excitation in [('nothing lit (the default)', None),
                          ('order 0 only', {0: 1.0}),
                          ('orders 0 and +1', {0: 1.0, 1: 0.5}),
                          ('orders -1, 0, +1', BEAMS),
                          ('orders +2 and +4', {2: 1.0, 4: 0.5})]:
    centre, half = window_of(excitation)
    print(f'  {label:26s} -> orders [{centre - half:+d}, {centre + half:+d}]'
          f'   ({2 * half + 1} of them)')

The extra orders are what make this *better* than solving beam by beam rather than merely
equal to it — measured, from `truncation_order` 8 up, on about a third of the orders the per-beam
route needs. They cost 3.6–9.1%, and the share falls as `truncation_order` grows because the
addition is a fixed number of orders.

The lit orders themselves must stay inside the declared `truncation_order`. That is what bounds the
widening: without it, `{0: 1.0, 9: 0.5}` at `truncation_order=6` would quietly retain twenty-two
orders instead of saying that 9 is outside what was asked for.

---

## 6. The one refusal: every lit order must propagate

An order past the light line (`|kx| > n_inc`) has an imaginary `kz` and carries no
time-averaged power. Lighting one would drive the structure while contributing nothing to
the incident flux every efficiency is divided by — so `R + T` climbs above one on a
perfectly passive stack, and "efficiency" quietly stops meaning *fraction of incident
power*.

It is refused instead, one order at a time:

In [ ]:
base_solver, _ = solve()
kx, _ = base_solver.order_kxy()
(centre, half), _ = base_solver.order_window
orders = np.arange(-half, half + 1) + centre
kx = np.real(np.asarray(kx))
print('order   kx      propagating in air?')
for m, value in zip(orders, kx):
    print(f'  {m:+d}   {value:+.3f}      {"yes" if abs(value) < 1.0 else "NO"}')

print()
try:
    solve({0: 1.0, 5: 0.5})
except ValueError as error:
    print(error)

The same fact caps how many beams can be combined: only about `2 * period / wavelength`
orders are inside the light line, so a finer grating combines fewer beams. That is physics,
not a limit of the code.

An evanescent *input* is a near-field source, which is a different problem than an incident
plane wave and one this library does not model.

---

## 7. A different polarization per order

`excitation` carries an amplitude per order and **one polarization for all of them** —
whatever `psi` / `polarization` / `tetm` said, each in its own order's s/p frame. For a
different polarization per order, go through the scattering matrix instead.

On the coupled paths (conical and 2D) the incident column is `[s-block ; p-block]`, each
as long as the order count, so filling the two halves independently *is* per-order
polarization. The s half is the s amplitude itself. The p half is a **port variable** -- the
p wave's magnetic field, `-1j * (n/mu) * a_p` in the lit medium -- and so is what comes back,
turned over for the up-going wave. It needs `layer_algorithm='SMM'`, which is the algorithm
that builds a scattering matrix:

In [ ]:
solver, _ = solve(algorithm='SMM', phi=0.3, tetm=None)
S = solver.scattering
(centre, half), _ = solver.order_window
n_orders = 2 * half + 1
print('S11 is', S.S11.shape, '-- 2 x', n_orders, 'because s and p are stacked')

h = 1.0                                                  # n/mu of the lit medium (air)
column = np.zeros((2 * n_orders, 1), dtype=complex)
column[0 - centre + half] = 1.0                          # order 0, s, amplitude 1
column[n_orders + 1 - centre + half] = -1j * h * (0.6 - 0.8j)   # order +1, p, in the port variable

reflected = (np.asarray(S.S11) @ column).ravel()
r_s = reflected[:n_orders]                               # s amplitudes as they are
r_p = reflected[n_orders:] / (1j * h)                    # p amplitudes, out of the port variable
print('\n  reflected |r_s| :', np.abs(r_s).round(4))
print('  reflected |r_p| :', np.abs(r_p).round(4))
print('\n  both halves respond, from one solve')

Two things to know about this route:

- **It hands back amplitudes, not efficiencies.** You normalize by whatever incident flux
  your column carries. `excitation` does that for you, which is most of what it is for.
- **`layer_algorithm='SMM'` only.** The ETM never forms a scattering matrix — that is why
  it is the cheaper default — and asking for `solver.scattering` after one says so.

With a *common* polarization the two routes agree to ~1e-16 on both halves -- the p half
once it is read as the port variable above (`S11` returns `-r_p` per unit of it, and `S21`
returns `t_p` times the exit medium's `n/mu` over the lit one's) -- so they are one contract
seen from two sides. `excitation` is the high-level door; the matrix is the low one.

---

## Where to read more

| | |
| --- | --- |
| [`../docs/multimode-incidence.md`](../docs/multimode-incidence.md) | the measurements: why the accuracy is a tie, why the order window stays symmetric, and what S4, fmmax and RETICOLO do |
| [`../docs/connecting-algorithms.md`](../docs/connecting-algorithms.md) | `solver.scattering`, and cascading two stacks through it |
| [`../docs/conventions.md`](../docs/conventions.md) | `kx`, `kz`, and what "incidence-relative" means |
| [`06-polarization-pair.ipynb`](06-polarization-pair.ipynb) | the polarization every lit order shares |